# Step 11: SP vs µP Comparison & Scaling Law Extrapolation
**Prerequisites**: SP results (`all_results.json`, `sweep_results.json`) and
µP results from steps 09-10. All expected on Google Drive.

In [5]:
import json, shutil
import numpy as np
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path("/content/drive/MyDrive/ML_project/checkpoints")
OUT = Path("/content/ML_project/checkpoints/comparison"); OUT.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
def load_results(path):
    with open(path) as f: data = json.load(f)
    data.sort(key=lambda r: r["n_params"])
    names = [r["model_name"] for r in data]
    params = np.array([r["n_params"] for r in data], dtype=float)
    losses = np.array([r["best_val_loss"] for r in data], dtype=float)
    return names, params, losses, data

sp_names, sp_params, sp_losses, sp_data = load_results(DRIVE/"all_results.json")
mup_names, mup_params, mup_losses, mup_data = load_results(DRIVE/"mup_scaling"/"all_results.json")

print("SP results:")
for n, p, l in zip(sp_names, sp_params, sp_losses): print(f"  {n:<8} {p/1e6:>8.1f}M  loss={l:.4f}")
print("\nmuP results:")
for n, p, l in zip(mup_names, mup_params, mup_losses): print(f"  {n:<8} {p/1e6:>8.1f}M  loss={l:.4f}")

SP results:
  tiny          1.3M  loss=1.8149
  small         3.4M  loss=1.6527
  medium       12.2M  loss=1.5971
  large        33.6M  loss=1.6307
  xl           88.1M  loss=1.8391

muP results:
  tiny          1.3M  loss=1.6485
  small         3.4M  loss=1.6461
  medium       12.2M  loss=1.6187
  large        33.6M  loss=1.6991
  xl           88.1M  loss=1.8295


In [8]:
def power_law(N, a, alpha, c): return a * np.power(N, -alpha) + c

def fit_pl(params, losses, label):
    try:
        popt, pcov = curve_fit(power_law, params, losses,
            p0=[10.0, 0.1, min(losses)*0.9],
            bounds=([0,0,0],[1e6,2.0,max(losses)]), maxfev=10000)
        a, alpha, c = popt; perr = np.sqrt(np.diag(pcov))
        y_pred = power_law(params, *popt)
        r2 = 1 - np.sum((losses-y_pred)**2)/np.sum((losses-np.mean(losses))**2)
        print(f"\n{label}: L = {a:.4f} * N^(-{alpha:.4f}) + {c:.4f}  [R²={r2:.4f}]")
        print(f"  alpha = {alpha:.4f} ± {perr[1]:.4f}, c = {c:.4f} ± {perr[2]:.4f}")
        return {"a":a,"alpha":alpha,"c":c,"popt":popt.tolist(),"pcov":pcov.tolist(),"r2":r2,"perr":perr.tolist()}, True
    except Exception as e:
        print(f"{label} fit failed: {e}"); return {}, False

sp_fit, sp_ok = fit_pl(sp_params, sp_losses, "SP")
mup_fit, mup_ok = fit_pl(mup_params, mup_losses, "muP")


SP: L = 1000000.0000 * N^(-1.1364) + 1.6746  [R²=0.1848]
  alpha = 1.1364 ± 5.0948, c = 1.6746 ± 0.1156

muP: L = 2.1736 * N^(-2.0000) + 1.6884  [R²=-0.0000]
  alpha = 2.0000 ± 0.0000, c = 1.6884 ± 0.0532


In [9]:
fig, ax = plt.subplots(figsize=(10,7))
ax.scatter(sp_params, sp_losses, s=120, c='#2196F3', zorder=5, edgecolors='white', lw=2, label='SP')
ax.scatter(mup_params, mup_losses, s=120, c='#E53935', zorder=5, edgecolors='white', lw=2, marker='D', label='muP')
for n,p,l in zip(sp_names, sp_params, sp_losses): ax.annotate(f"  {n}", (p,l), fontsize=8, color='#2196F3')
for n,p,l in zip(mup_names, mup_params, mup_losses): ax.annotate(f"  {n}", (p,l), fontsize=8, color='#E53935')

for fit, ok, col, lbl in [(sp_fit, sp_ok, '#2196F3', 'SP'), (mup_fit, mup_ok, '#E53935', 'muP')]:
    if ok:
        xf = np.logspace(np.log10(min(sp_params.min(),mup_params.min())*0.5),
                         np.log10(max(sp_params.max(),mup_params.max())*2), 200)
        ax.plot(xf, power_law(xf, *fit["popt"]), '--', color=col, lw=2, alpha=0.7,
                label=f'{lbl}: α={fit["alpha"]:.4f}, R²={fit["r2"]:.4f}')
ax.set_xscale('log'); ax.set_xlabel("Parameters", fontsize=13); ax.set_ylabel("Val Loss", fontsize=13)
ax.set_title("Scaling Laws: SP vs muP", fontsize=15, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, which='both'); plt.tight_layout()
plt.savefig(OUT/"sp_vs_mup_scaling.png", dpi=150); plt.show(); plt.close()

In [10]:
fig, ax = plt.subplots(figsize=(8,5))
sp_sw = DRIVE/"lr_sweep"/"sweep_results.json"
mup_sw = DRIVE/"mup_lr_sweep"/"sweep_results.json"
if sp_sw.exists():
    with open(sp_sw) as f: sw = json.load(f)
    lrs = [r["lr"] for r in sw["results"]]; vl = [r["best_val_loss"] for r in sw["results"]]
    ax.semilogx(lrs, vl, 'o-', color='#2196F3', ms=8, lw=2, label=f'SP (best: {sw["best_lr"]:.1e})')
if mup_sw.exists():
    with open(mup_sw) as f: sw = json.load(f)
    valid = [(r["lr"],r["best_val_loss"]) for r in sw["results"] if r["best_val_loss"] < 1e6]
    if valid:
        lrs, vl = zip(*valid)
        ax.semilogx(lrs, vl, 'D-', color='#E53935', ms=8, lw=2, label=f'muP (best: {sw["best_lr"]:.1e})')
ax.set_xlabel("Learning Rate"); ax.set_ylabel("Best Val Loss")
ax.set_title("LR Sweep: SP vs muP (Tiny)", fontsize=14, fontweight='bold')
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout()
plt.savefig(OUT/"lr_sweep_comparison.png", dpi=150); plt.show(); plt.close()

In [11]:
fig, axes = plt.subplots(1, 2, figsize=(16,6))
colors = ['#4CAF50','#2196F3','#FF9800','#E53935','#9C27B0']
for i, (sr, mr, c) in enumerate(zip(sp_data, mup_data, colors)):
    n = sr["model_name"]
    if sr.get("val_losses"):
        s,l = zip(*sr["val_losses"]); axes[0].plot(s,l,'o-',color=c,lw=1.5,ms=4,label=f'{n} ({sr["n_params"]/1e6:.1f}M)')
    if mr.get("val_losses"):
        s,l = zip(*mr["val_losses"]); axes[1].plot(s,l,'D-',color=c,lw=1.5,ms=4,label=f'{n} ({mr["n_params"]/1e6:.1f}M)')
for i, t in enumerate(["SP Validation Curves", "muP Validation Curves"]):
    axes[i].set_xlabel("Step"); axes[i].set_ylabel("Val Loss"); axes[i].set_title(t, fontweight='bold')
    axes[i].legend(fontsize=9); axes[i].grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(OUT/"training_curves_comparison.png", dpi=150); plt.show(); plt.close()

In [12]:
fig, ax = plt.subplots(figsize=(10,6))
x = np.arange(len(sp_names)); w = 0.35
b1 = ax.bar(x-w/2, sp_losses, w, label='SP', color='#2196F3', alpha=0.8)
b2 = ax.bar(x+w/2, mup_losses, w, label='muP', color='#E53935', alpha=0.8)
for b in [b1, b2]:
    for bar in b: ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01, f'{bar.get_height():.3f}', ha='center', fontsize=9)
ax.set_xlabel("Model"); ax.set_ylabel("Best Val Loss")
ax.set_title("SP vs muP: Per-Model", fontsize=14, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels([f"{n}\n({p/1e6:.1f}M)" for n,p in zip(sp_names, sp_params)])
ax.legend(); ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout()
plt.savefig(OUT/"per_model_comparison.png", dpi=150); plt.show(); plt.close()

In [13]:
print(f"\n{'='*60}\nSCALING LAW EXTRAPOLATION\n{'='*60}")
if sp_ok and mup_ok:
    bl = "muP" if mup_fit["r2"] > sp_fit["r2"] else "SP"
    bf = mup_fit if bl == "muP" else sp_fit
    bp = mup_params if bl == "muP" else sp_params
elif mup_ok: bl, bf, bp = "muP", mup_fit, mup_params
elif sp_ok: bl, bf, bp = "SP", sp_fit, sp_params
else: raise RuntimeError("No valid fits!")

a, alpha, c = bf["popt"]; pcov = np.array(bf["pcov"])
xl_p = bp.max(); t10x = xl_p * 10
pred = power_law(t10x, a, alpha, c)
J = np.array([t10x**(-alpha), -a*t10x**(-alpha)*np.log(t10x), 1.0])
ci95 = 1.96 * np.sqrt(max(J @ pcov @ J, 0))

print(f"Best fit: {bl} (R²={bf['r2']:.4f})")
print(f"XL params: {xl_p/1e6:.1f}M")
print(f"10x target: {t10x/1e6:.1f}M")
print(f"Predicted loss: {pred:.4f} ± {ci95:.4f} (95% CI)")
print(f"95% CI: [{pred-ci95:.4f}, {pred+ci95:.4f}]")

for mult, label in [(2,"2x"),(5,"5x"),(10,"10x")]:
    t = xl_p*mult; p = power_law(t, a, alpha, c)
    print(f"  {label} ({t/1e6:.1f}M): {p:.4f}")

# Extrapolation plot
fig, ax = plt.subplots(figsize=(10,7))
bl_losses = mup_losses if bl == "muP" else sp_losses
ax.scatter(bp, bl_losses, s=120, c='#2196F3', zorder=5, edgecolors='white', lw=2, label=f'{bl} trained')
xt = np.logspace(np.log10(bp.min()*0.5), np.log10(bp.max()*1.1), 200)
ax.plot(xt, power_law(xt, a, alpha, c), '-', color='#2196F3', lw=2, alpha=0.8)
xe = np.logspace(np.log10(bp.max()), np.log10(t10x*1.5), 200)
ax.plot(xe, power_law(xe, a, alpha, c), '--', color='#FF9800', lw=2, label='Extrapolation')
yu, yl = [], []
for N in xe:
    Jn = np.array([N**(-alpha), -a*N**(-alpha)*np.log(N), 1.0])
    ci = 1.96*np.sqrt(max(Jn @ pcov @ Jn, 0)); pn = power_law(N, a, alpha, c)
    yu.append(pn+ci); yl.append(pn-ci)
ax.fill_between(xe, yl, yu, alpha=0.15, color='#FF9800', label='95% CI')
ax.scatter([t10x], [pred], s=200, c='#E53935', zorder=6, marker='*', edgecolors='white', lw=2,
           label=f'10x: {pred:.4f}')
ax.set_xscale('log'); ax.set_xlabel("Parameters"); ax.set_ylabel("Val Loss")
ax.set_title(f"Scaling Law Extrapolation ({bl})", fontsize=15, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3, which='both'); plt.tight_layout()
plt.savefig(OUT/"extrapolation_plot.png", dpi=150); plt.show(); plt.close()


SCALING LAW EXTRAPOLATION
Best fit: SP (R²=0.1848)
XL params: 88.1M
10x target: 881.0M
Predicted loss: 1.6747 ± 0.2233 (95% CI)
95% CI: [1.4514, 1.8980]
  2x (176.2M): 1.6750
  5x (440.5M): 1.6748
  10x (881.0M): 1.6747


In [14]:
DRIVE_OUT = Path("/content/drive/MyDrive/ML_project/checkpoints/comparison")
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

summary = {"sp_fit": sp_fit if sp_ok else None, "mup_fit": mup_fit if mup_ok else None,
           "best_approach": bl,
           "extrapolation": {"xl_params": float(xl_p), "target_10x": float(t10x),
                             "predicted_loss": float(pred), "ci_95": float(ci95)},
           "sp_results": {n: {"params": float(p), "val_loss": float(l)} for n,p,l in zip(sp_names, sp_params, sp_losses)},
           "mup_results": {n: {"params": float(p), "val_loss": float(l)} for n,p,l in zip(mup_names, mup_params, mup_losses)}}
with open(OUT/"comparison_summary.json", 'w') as f: json.dump(summary, f, indent=2)

for f in ["sp_vs_mup_scaling.png", "lr_sweep_comparison.png", "training_curves_comparison.png",
          "per_model_comparison.png", "extrapolation_plot.png", "comparison_summary.json"]:
    shutil.copy2(OUT/f, DRIVE_OUT/f)

print(f"\n{'='*60}\nSP vs muP COMPARISON\n{'='*60}")
print(f"{'Model':<8} {'SP':>10} {'muP':>10} {'Diff':>10} {'Better?':>8}")
print("-"*50)
for n, sl, ml in zip(sp_names, sp_losses, mup_losses):
    d = ml - sl; print(f"{n:<8} {sl:>10.4f} {ml:>10.4f} {d:>+10.4f} {'muP' if d<0 else 'SP':>8}")
if sp_ok and mup_ok:
    print(f"\nSP α={sp_fit['alpha']:.4f}, muP α={mup_fit['alpha']:.4f}")
    print(f"Steeper: {'muP' if mup_fit['alpha']>sp_fit['alpha'] else 'SP'}")
print(f"\nAll plots saved to Drive: {DRIVE_OUT}")


SP vs muP COMPARISON
Model            SP        muP       Diff  Better?
--------------------------------------------------
tiny         1.8149     1.6485    -0.1664      muP
small        1.6527     1.6461    -0.0067      muP
medium       1.5971     1.6187    +0.0216       SP
large        1.6307     1.6991    +0.0684       SP
xl           1.8391     1.8295    -0.0097      muP

SP α=1.1364, muP α=2.0000
Steeper: muP

All plots saved to Drive: /content/drive/MyDrive/ML_project/checkpoints/comparison
